In [ ]:
# ============================================================
# ProjectTwoDashboard.ipynb
# Author: Freddy Case
# Version: 2.0
# Date: October 4th, 2026
# Original: CS 340, December 2025
#
# Purpose:
# Interactive dashboard for Grazioso Salvare that filters AAC shelter
# animals by rescue type and shows the results in a table, breed chart,
# and map.
#
# Changes in version 2.0 (CS 499 enhancement):
# - Included a login with a read-only MongoDB account loaded from a .env file
# - Now breed chart counts with a MongoDB aggregation pipeline
# - Rescue filters now share one build_query() function
# - Map now looks up columns by name instead of column number
# - Replaced the deprecated jupyter_dash package with Dash's built-in
#   Jupyter support
# ============================================================

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
# added in import dash
import dash
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
from dotenv import load_dotenv

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
#import matplotlib.pyplot as plt

# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Load environment variables from .env file
# .env file is listed in .gitignore to prevent it from being uploaded to GitHub
load_dotenv()
username = os.getenv("AAC_ADMIN_USER")
password = os.getenv("AAC_ADMIN_PASS")

# Stop right away with a clear message if the credentials are missing
if not username or not password:
    raise ValueError("Missing AAC_USER or AAC_PASS in the .env file")

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Reload the environment variables to ensure the AAC Reader credentials are available
load_dotenv()

# Load environment variables for the AAC Reader user
username = os.getenv("AAC_READ_USER")
password = os.getenv("AAC_READ_PASS")

# Check if the AAC Reader credentials are available
if not username or not password:
    raise ValueError("Missing AAC_READ_USER or AAC_READ_PASS in the .env file")

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = dash.Dash(__name__)

image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# Give the dashboard its own light background and dark text. Without a
# background color the page is see through.
PAGE_STYLE = {
    'backgroundColor': '#ffffff',
    'color': '#222222',
    'padding': '20px',
    'fontFamily': 'Arial, sans-serif',
}

app.layout = html.Div([
    html.Div([
        # Client logo encoded in Base64 so it u uploads directly in the Dash
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '60px'}
            ),
        ),
        # Dashboard title
        html.Span("  Grazioso Salvare Dashboard - Freddy Case",
                  style={'fontSize': '24px', 'fontWeight': 'bold', 'marginLeft': '15px'})
    # Align logo and title on one row
    ], style={'display': 'flex', 'alignItems': 'center'}),
#    html.Div(id='hidden-div', style={'display':'none'}),
    # Center dashboard
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),
   # html.Div(
        html.Div([
        html.Label('Rescue Type Filter'),
        # Add radio buttons
        dcc.RadioItems(
            id='filter-type',
            # Each type of button
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
             # Default setting loads entire dataset
            value='reset',
                        labelStyle={'display': 'inline-block', 'marginRight': '15px', 'color': '#222222'}
        )
    ], style={'textAlign': 'center'}),

    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         editable=False,
                         filter_action="native",
                         sort_action="native",
                         sort_mode="multi",
                         column_selectable=False,
                         row_selectable="single",
                         row_deletable=False,
                         selected_rows=[0],
                         selected_columns=[],
                         page_action="native",
                         page_current=0,
                         page_size=10,  
                        ),
    html.Br(),
    html.Hr(),
    # Sets up the dashboard so that the chart and geolocation are side by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
], style=PAGE_STYLE)

#############################################
# Interaction Between Components / Controller
#############################################

# build_query method builds the MongoDB query based on the selected filter type
def build_query(filter_type):
    """
    Returns the MongoDB query for the selected rescue type. Used by both
    the data table and the breed chart so they always filter the same way.
    """
    if filter_type == 'water':
        return {
            'animal_type': 'Dog',
            'breed': {'$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']},
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    if filter_type == 'mountain':
        return {
            'animal_type': 'Dog',
            'breed': {'$in': ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog',
                              'Siberian Husky', 'Rottweiler']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    if filter_type == 'disaster':
        return {
            'animal_type': 'Dog',
            'breed': {'$in': ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever',
                              'Bloodhound', 'Rottweiler']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
        }
    # Reset: no filter, return everything
    return {}

# Callback to update the data table based on the selected filter type
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    global df
    # Build the MongoDB query based on the selected filter type
    query = build_query(filter_type)
    # Use MongoDB with the Crud module
    df = pd.DataFrame.from_records(db.read(query))

    # Removes MongoDB_id to avoid rendering issues
    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)

    # convert the results into the list of rows the data table expects
    data = df.to_dict('records')

    return data

# Callback to update the pie chart based on the selected rescue type
@app.callback(
    Output('graph-id', "children"),
    [Input('filter-type', 'value')])
def update_graphs(filter_type):
    breed_counts = db.get_breed_counts(build_query(filter_type))

    # If nothing matches the filter, there is nothing to return.
    if not breed_counts:
        return [html.P("No matching animals to chart.")]

    # MongoDB names the grouped field "_id", so rename it to "breed"
    chart_df = pd.DataFrame(breed_counts).rename(columns={"_id": "breed"})

    # return the pie chart with the breed counts.
    return [
        dcc.Graph(
            figure=px.pie(chart_df, names='breed', values='count', title='Preferred Animals')
        )
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # look up each value by column name instead of column number, since the
    # number of columns changes when some animals are missing a field
    selected = dff.iloc[row]
    breed = selected.get('breed', 'Unknown breed')
    name = selected.get('name')

    # some animals have no name on record
    if name is None or pd.isna(name):
        name = "No name on record"

    # Create the map with the selected animal's location and details.
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # marker placed at the selected animal's location
            dl.Marker(position=[selected['location_lat'], selected['location_long']], children=[
                dl.Tooltip(breed),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(name)
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
# To get it to build, changing run_server to run and the parameter mode to jupyter_mode.
# app.run_server(mode='inline', port=8051)
app.run(jupyter_mode='inline', port=8051)

In [ ]:
# Shows how MongoDB runs the water rescue query, and whether it uses
# rescue_filter_index from setup_indexes.py, or reads every document.
water_query = {
    'animal_type': 'Dog',
    'breed': {'$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']},
    'sex_upon_outcome': 'Intact Female',
    'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
}
stats = db.collection.find(water_query).explain()["executionStats"]
print("Matches returned:", stats["nReturned"])
print("Documents examined:", stats["totalDocsExamined"])
print("Index keys examined:", stats["totalKeysExamined"])